# Loader checkpoint experiment
Runs the production loader recovery code with isolated toy inputs. Three CSV rows are strings; no PDF or database is created.

Flow: ingestion output → loader proposes code → fresh process executes → error goes to glitch → loader repairs → Parquet/manifest validation → review boundary. The real loader contracts, prompts, manifest validator and chunked rollover function are imported.

This notebook uses **scripted API replies**, with the fake remote server kept separately under `test_server`. Its records are test instrumentation. Application recovery reads only `work/checkpoint.json`, current execution files and prepared artifacts. The notebook and app use the same LoaderRunner; this notebook touches only its toy workspace.

The proposed change is saving `result` and `code_ran=True` together **before** calling glitch. If glitch has no saved response, retry diagnosis in its saved thread; otherwise reuse it. Advancing the loader overwrites the previous cycle.

Configured requests match the app: loader/glitch Sol medium, rollover Luna high; 250,000 session tokens and 8,000 loader output tokens. The fake token counter deliberately forces one rollover. This notebook makes no paid model calls.

Only trusted fixture scripts use the fast local-process adapter here. The test command with `--docker` exercises the unchanged app Docker runner and its real limits. Reviews and later codebook/SQL publication are outside this loader experiment.

In [ ]:
from pathlib import Path
import sys
repo = Path.cwd() if (Path.cwd() / 'v_5').is_dir() else Path.cwd().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from v_5.loader_prototype import LAB
from v_5.tests.loader_fixtures import build_case
from v_5.prototype_runtime import InjectedFailure
from v_5.ingest.codebook_ingestion.sub_agents.pdf_state import read_json
case = LAB / 'app_notebook'
plan = [
    {'point': 'after_execution_result', 'unit': '1'},
    {'point': 'after_glitch_submit', 'unit': '1'},
    {'point': 'after_glitch_response', 'unit': '1'},
    {'point': 'after_rollover_summary', 'unit': '2'},
]


In [ ]:
for attempt in range(len(plan) + 1):
    runner, api = build_case(case, plan=plan)
    try:
        final = runner.run()
        print('Finished:', final['phase'])
        break
    except InjectedFailure as error:
        saved = read_json(case / 'work' / 'checkpoint.json')
        loader = saved['loader']
        print('Interrupted:', error)
        print({key: loader[key] for key in ('cycle', 'input_sent', 'loader_returned', 'code_ran')})
        print('Saved glitch response:', bool((loader['glitch'] or {}).get('response')))
else:
    raise RuntimeError('The experiment did not recover')


In [ ]:
import pandas as pd
events = read_json(api.path)['events']
assert sum(item['kind'] == 'execute' for item in events) == 2
assert sum(item['kind'] == 'ingestion' for item in events) == 1
table = pd.read_parquet(case / 'work' / 'toy.parquet')
assert table['id'].tolist() == [1, 2, 3]
print('The failed script ran once; the corrected script ran once. All three rows survived.')
print('Checkpoint:', case / 'work' / 'checkpoint.json')
table


Repeatable stress suite, from the repository root:
```text
python -B -m v_5.tests.loader_checkpoint_check
python -B -m v_5.tests.loader_checkpoint_check --docker
```
A crash after execution but before the result is saved can repeat that script. That is the agreed tradeoff: the fixtures overwrite their outputs, and tests verify no duplicated rows. A saved execution error is never rerun merely because glitch failed.

The app now uses this same checkpoint runner. Retry preparation reopens its saved workspace and API threads. Changed inputs or a fresh material review clear that workspace pointer.